In [1]:
!pip install pandas sentence-transformers faiss-cpu langchain chromadb tqdm python-dotenv
!pip install langchain-community


In [2]:
!pip install langchain-ollama

In [3]:
import pandas as pd
from langchain_core.documents import Document

# ubah path jika perlu
df = pd.read_excel("chatbot_its_dataset.xlsx")
df.head()

# Tampilkan kolom agar tahu nama kolom persis
print("Columns:", df.columns.tolist())

# Sesuaikan nama kolom di bawah bila berbeda
q_col = "question" if "question" in df.columns.str.lower() else df.columns[0]
a_col = "answer" if "answer" in df.columns.str.lower() else df.columns[1]

# safe mapping in case columns have different exact names:
q_col = df.columns[0]
a_col = df.columns[2]

print("Using columns:", q_col, "->", a_col)

# Create knowledge chunks: store both Q and A as context
documents = []
metas = []
for i, row in df.iterrows():
    q = str(row[q_col])
    a = str(row[a_col])
    text = f"FAQ Question: {q}\nReference Answer: {a}"
    documents.append(Document(page_content=text, metadata={"source_row": int(i)}))


Columns: ['question', 'intent', 'response']
Using columns: question -> response


In [4]:
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss
import pickle
from pathlib import Path

# config
EMB_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
INDEX_DIR = "./stored"
INDEX_PATH = f"{INDEX_DIR}/faiss_index.idx"
CHUNKS_PATH = f"{INDEX_DIR}/chunks.pkl"
METAS_PATH = f"{INDEX_DIR}/metas.pkl"

Path(INDEX_DIR).mkdir(parents=True, exist_ok=True)

# embed function (batch)
embed_model = SentenceTransformer(EMB_MODEL)

texts = [doc.page_content for doc in documents]
print("Total chunks:", len(texts))

# compute embeddings (batch)
embeddings = embed_model.encode(texts, show_progress_bar=True, convert_to_numpy=True)

# build FAISS index
dim = embeddings.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(embeddings)
faiss.write_index(index, INDEX_PATH)

# save chunks & metas
with open(CHUNKS_PATH, "wb") as f:
    pickle.dump(texts, f)
with open(METAS_PATH, "wb") as f:
    pickle.dump([doc.metadata for doc in documents], f)

print("Saved FAISS index, chunks, metas in", INDEX_DIR)


C:\Users\raras\anaconda3\envs\openllm\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Total chunks: 20


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.88it/s]

Saved FAISS index, chunks, metas in ./stored


In [5]:
import pickle

# load index & chunks
index = faiss.read_index(INDEX_PATH)
with open(CHUNKS_PATH, "rb") as f:
    chunks = pickle.load(f)
with open(METAS_PATH, "rb") as f:
    metas = pickle.load(f)

def retrieve(query, k=3):
    q_emb = embed_model.encode([query], convert_to_numpy=True)
    D, I = index.search(q_emb, k)
    results = []
    for idx in I[0]:
        results.append({"text": chunks[idx], "metadata": metas[idx]})
    return results


In [6]:
from langchain_ollama import OllamaLLM

# jika Ollama serve default di localhost:11434, wrapper akan memakai itu
llm = OllamaLLM(model="llama3.1")  # ganti model sesuai yang sudah kamu pull
def generate_from_llm(prompt, max_tokens=256):
    return llm.invoke(prompt)


In [7]:
from textwrap import shorten

SYSTEM_INSTR = (
    "Kamu adalah asisten resmi yang menjawab pertanyaan terkait proses akademik ITS. "
    "Berikan jawaban terstruktur dengan format: "
    "1. Penjelasan ringkas "
    "2. Detail persyaratan / prosedur "
    "3. Catatan tambahan / tips penting "

    "Gunakan konteks sebagai sumber data faktual, tetapi jangan menyalin kalimat secara langsung."
    "Susun ulang dengan bahasa informatif dan lebih lengkap. "
    "Jika konteks tidak cukup untuk menjawab, bilang 'Maaf, informasi tidak tersedia dalam sumber.'"
)

def build_prompt(user_q, retrieved):
    # build a context string listing retrieved items with small excerpts
    context_blocks = []
    for i, r in enumerate(retrieved, 1):
        # limit length to avoid too long prompt
        excerpt = shorten(r["text"], width=600, placeholder="...")
        context_blocks.append(f"[{i}] {excerpt}")
    context = "\n\n".join(context_blocks)
    prompt = (
        SYSTEM_INSTR + "\n\n"
        f"Context (reference QA pairs):\n{context}\n\n"
        f"Pertanyaan pengguna: {user_q}\n\n"
        "Gunakan konteks di atas untuk menyusun jawaban baru (jangan menambahkan fakta yang tidak ada). "
        "Cantumkan juga sumber dalam format: Sumber: [index]."
    )
    return prompt

def rag_answer(user_q, k=3):
    retrieved = retrieve(user_q, k=k)
    prompt = build_prompt(user_q, retrieved)
    # call LLM
    llm_out = generate_from_llm(prompt)
    # llm_out may be LangChain object/string; convert to string if needed
    ans = llm_out if isinstance(llm_out, str) else str(llm_out)
    return {"answer": ans, "retrieved": retrieved, "prompt": prompt}


In [8]:
print("RAG chatbot siap. Ketik 'exit' untuk keluar.")
while True:
    q = input("Tanya: ").strip()
    if q.lower() in ("exit","quit"):
        break
    res = rag_answer(q, k=3)
    print("\nJawaban:\n", res["answer"])
    print("\nSumber referensi:")
    for i, r in enumerate(res["retrieved"],1):
        print(f"[{i}] row={r['metadata'].get('source_row')}  snippet: {r['text'][:200]}...")
    print("\n---\n")


RAG chatbot siap. Ketik 'exit' untuk keluar.


Tanya:  Bagaimana cara membayar uang kuliah tunggal semester ini?



Jawaban:
 1. Penjelasan Ringkas
Membayar uang kuliah tunggal semester ini dapat dilakukan melalui portal SIM Akademik.

2. Detail Persyaratan / Prosedur
- Masuk ke portal SIM Akademik.
- Pilih menu pembayaran atau sesuai dengan label lainnya terkait pembayaran di halaman tersebut.
- Proses pembayaran akan disajikan, termasuk rincian biaya yang harus dibayarkan.

3. Catatan Tambahan / Tips Penting
Pastikan telah login menggunakan akun pengguna yang aktif dan valid sebelum memulai proses pembayaran.

Sumber referensi:
[1] row=4  snippet: FAQ Question: Bagaimana cara mengisi FRS?
Reference Answer: FRS diisi melalui portal SIM Akademik di awal semester....
[2] row=13  snippet: FAQ Question: Bagaimana cara melihat nilai perkuliahan saya?
Reference Answer: Nilai dari masing-masing mata kuliah dapat dilihat di MyITS Academics....
[3] row=11  snippet: FAQ Question: Bagaimana cara mengakses myITS Classroom?
Reference Answer: myITS Classroom dapat diakses di MyITS SSO (my.its.ac.id). Setelah an

Tanya:  exit
